In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

DEBUG = False   

import glob, json, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchvision
from torchvision.transforms import v2
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

assert torch.cuda.is_available(), "No GPU found. Set Accelerator to GPU T4 x2 in Session options."
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

hits = glob.glob("/kaggle/input/**/images_384.npy", recursive=True)
assert hits, "images_384.npy not found. Add the output of xray-01-data-check as an input."
SRC = os.path.dirname(hits[0])
print("Loading from:", SRC)

images = np.load(f"{SRC}/images_384.npy")
df = pd.read_csv(f"{SRC}/splits.csv")
assert len(images) == len(df) and (df.idx.values == np.arange(len(df))).all(), "Images and table do not line up."
print("Images:", images.shape, "| patients per split:", df.split.value_counts().to_dict())

PyTorch 2.11.0+cu128 | GPU: Tesla T4
Loading from: /kaggle/input/notebooks/divinembah/xray-01-data-check
Images: (26684, 384, 384) | patients per split: {'train': 18678, 'test': 5337, 'val': 2669}


In [3]:
BATCH = 32
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()   
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

augment = v2.Compose([
    v2.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.9, 1.1)),   
    v2.ColorJitter(brightness=0.15, contrast=0.15),                     
])

class XrayData(torch.utils.data.Dataset):
    def __init__(self, frame, train):
        self.idx = frame.idx.values
        self.y = frame.target.values.astype("float32")
        self.train = train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        x = torch.from_numpy(images[self.idx[i]]).unsqueeze(0) 
        if self.train:
            x = augment(x)
        return x, self.y[i]

def prep(x):
    """Move a batch to the GPU, scale to 0-1, copy grey to 3 channels, normalise."""
    x = x.cuda(non_blocking=True).float() / 255.0
    return (x.expand(-1, 3, -1, -1) - MEAN) / STD

parts = {s: df[df.split == s] for s in ["train", "val", "test"]}
if DEBUG:
    parts = {s: p.sample(640 if s == "train" else 320, random_state=SEED) for s, p in parts.items()}

def loader(split, train):
    return torch.utils.data.DataLoader(XrayData(parts[split], train), batch_size=BATCH, shuffle=train,
                                       drop_last=train, num_workers=4, pin_memory=True)

train_dl, val_dl, test_dl = loader("train", True), loader("val", False), loader("test", False)

x, y = next(iter(train_dl))
print("One batch:", tuple(x.shape), x.dtype, "| labels:", y[:8].tolist())
print("After prep:", tuple(prep(x).shape), "| mean", round(prep(x).mean().item(), 2))
print({s: len(p) for s, p in parts.items()})

One batch: (32, 1, 384, 384) torch.uint8 | labels: [0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0]
After prep: (32, 3, 384, 384) | mean 0.1
{'train': 18678, 'val': 2669, 'test': 5337}


In [4]:
EPOCHS = 1 if DEBUG else 6
LR = 1e-4

model = torchvision.models.densenet121(weights="IMAGENET1K_V1")       
model.classifier = nn.Linear(model.classifier.in_features, 1)      
model = model.cuda()

loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")

@torch.no_grad()
def predict(dl):
    """Return the model's pneumonia probability for every image in dl, and the average loss."""
    model.eval()
    probs, losses = [], []
    for x, y in dl:
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(prep(x)).squeeze(1)
        losses.append(loss_fn(logits.float(), y.cuda()).item() * len(y))
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(probs), sum(losses) / len(dl.dataset)

log, best_auc, start = [], 0.0, time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    running, seen = 0.0, 0
    for step, (x, y) in enumerate(train_dl, 1):
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(prep(x)).squeeze(1).float(), y.cuda())
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        running += loss.item() * len(y); seen += len(y)
        if step % 100 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_dl)} train loss {running / seen:.4f}")

    val_prob, val_loss = predict(val_dl)
    val_auc = roc_auc_score(parts["val"].target.values, val_prob)
    row = {"epoch": epoch, "train_loss": round(running / seen, 4), "val_loss": round(val_loss, 4),
           "val_auc": round(val_auc, 4), "minutes": round((time.time() - start) / 60, 1)}
    log.append(row); print(row)
    if val_auc > best_auc:                               
        best_auc = val_auc
        torch.save(model.state_dict(), "/kaggle/working/densenet121_rsna.pt")

json.dump(log, open("/kaggle/working/training_log.json", "w"), indent=2)
print("Best validation AUC:", round(best_auc, 4))

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 231MB/s]


  epoch 1 step 100/583 train loss 0.6064
  epoch 1 step 200/583 train loss 0.5099
  epoch 1 step 300/583 train loss 0.4683
  epoch 1 step 400/583 train loss 0.4466
  epoch 1 step 500/583 train loss 0.4356
{'epoch': 1, 'train_loss': 0.4289, 'val_loss': 0.3755, 'val_auc': np.float64(0.8699), 'minutes': 4.3}
  epoch 2 step 100/583 train loss 0.3523
  epoch 2 step 200/583 train loss 0.3460
  epoch 2 step 300/583 train loss 0.3485
  epoch 2 step 400/583 train loss 0.3529
  epoch 2 step 500/583 train loss 0.3522
{'epoch': 2, 'train_loss': 0.355, 'val_loss': 0.3506, 'val_auc': np.float64(0.8849), 'minutes': 8.7}
  epoch 3 step 100/583 train loss 0.3345
  epoch 3 step 200/583 train loss 0.3358
  epoch 3 step 300/583 train loss 0.3392
  epoch 3 step 400/583 train loss 0.3361
  epoch 3 step 500/583 train loss 0.3337
{'epoch': 3, 'train_loss': 0.3342, 'val_loss': 0.3343, 'val_auc': np.float64(0.8907), 'minutes': 13.0}
  epoch 4 step 100/583 train loss 0.2946
  epoch 4 step 200/583 train loss 0.29

In [5]:
model.load_state_dict(torch.load("/kaggle/working/densenet121_rsna.pt"))  

out = []
for s, dl in [("val", val_dl), ("test", test_dl)]:
    prob, _ = predict(dl)
    p = parts[s][["patientId", "split", "class", "target", "view", "sex", "age", "n_boxes", "idx"]].copy()
    p["prob"] = prob
    out.append(p)
    print(f"{s}: {len(p)} patients | ROC AUC {roc_auc_score(p.target, p.prob):.4f} | PR AUC {average_precision_score(p.target, p.prob):.4f}")

preds = pd.concat(out)
preds.to_csv("/kaggle/working/predictions.csv", index=False)
print("\nSaved:", sorted(os.listdir("/kaggle/working")))

val: 2669 patients | ROC AUC 0.8914 | PR AUC 0.7227
test: 5337 patients | ROC AUC 0.8840 | PR AUC 0.7145

Saved: ['__notebook__.ipynb', 'densenet121_rsna.pt', 'predictions.csv', 'training_log.json']
